## setup
```bash
conda env create -f environment.yml
conda activate cs229_md_project
python -m ipykernel install --user --name cs229_md --display-name "cs229_md"
jupyter notebook

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import pyemma.coordinates as coor
import subprocess
from collections import defaultdict
import warnings
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import shutil

# md stuff
import MDAnalysis as mda
from MDAnalysis.analysis import rms, alignimport os
from pathlib import Path
import numpy as np
import pandas as pd
import pyemma.coordinates as coor
import subprocess
from collections import defaultdict
import warnings
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import shutil

# md stuff
import MDAnalysis as mda
from MDAnalysis.analysis import rms, align

In [2]:
# == specify paths ===

# mdanalysis cannot read trajectories directly from GCS use workflow: GCS -> local scratch -> mdanalysis -> outputs
CACHE_DIR = Path("/home/jupyter/gcs_cache")  # on workbench
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# output directories for labeled data
OUT_DIR = Path("/home/jupyter/cs229-md-prediction/data/processed/v3")
OUT_DIR.mkdir(exist_ok=True)

# data file paths on GCS
BUCKET_DIR = "gs://cs229-central"
MODELS_DIR = f"{BUCKET_DIR}/data/models"
TOPOS_DIR   = f"{BUCKET_DIR}/data/topologies"
TRAJS_DIR   = f"{BUCKET_DIR}/data/trajectories"

print("CACHE_DIR:", CACHE_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())

# === helpers for automatic file parsing ===

def gs_ls(prefix):
    out = subprocess.check_output(["gsutil", "ls", "-r", f"{prefix}/**"], text=True)
    return [line.strip() for line in out.splitlines()
            if line.strip() and not line.strip().endswith("/")]
    
def base3(gs_path):
    # protein~pdbid~state~...
    name = Path(gs_path).name
    parts = name.split("~")
    return "~".join(parts[:3]) if len(parts) >= 3 else None

def base2(gs_path):
    # protein~pdbid_or_pdbid_chain
    parts = Path(gs_path).name.split("~")
    return "~".join(parts[:2]) if len(parts) >= 2 else None

def sim_id(gs_path):
    name = Path(gs_path).name
    m = re.search(r"(?:trj|dyn)_(?:\d+_)?(\d+)", name)
    return m.group(1) if m else None
    
# download gcs_path -> cache_dir if not already present and return local path
def gs_fetch(gs_path):
    local = CACHE_DIR / Path(gs_path).name
    if not local.exists():
        # suppress gsutil stdout/stderr so we don't get spam outputs
        subprocess.run(
            ["gsutil", "-m", "cp", gs_path, str(local)],
            check=True,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )
    return local

CACHE_DIR: /home/jupyter/gcs_cache
OUT_DIR: /home/jupyter/cs229-md-prediction/data/processed/v3


In [3]:
# === build traj_info dictionary (top, model, traj) ===

# # get all files from GCS directory
# traj_files  = [p for p in gs_ls(TRAJS_DIR)  if p.endswith((".xtc", ".trr", ".dcd"))]
# top_files   = [p for p in gs_ls(TOPOS_DIR)  if p.endswith((".psf", ".top", ".pdb"))]
# model_files = [p for p in gs_ls(MODELS_DIR) if p.endswith(".pdb")]

# # prepare containers for grouping files by system
# trajs = defaultdict(list)
# tops  = {}
# models  = {}

# for p in traj_files:
#     k = base3(p)
#     if k: trajs[k].append(p)

# def topo_rank(p):
#     if p.endswith(".psf"): return 0
#     if p.endswith(".top"): return 1
#     if p.endswith(".pdb"): return 2
#     return 99

# for p in top_files:
#     k = base3(p)
#     if not k:
#         continue
#     if (k not in tops) or (topo_rank(p) < topo_rank(tops[k])):
#         tops[k] = p

# for p in model_files:
#     k = base3(p)
#     if k and k not in models: models[k] = p

# traj_info = {}
# missing = []
# for k, trajs in trajs.items():
#     top = tops.get(k)
#     model = models.get(k)

#     # fallback: use whichever exists (model or top)
#     if top is None and model is None:
#         missing.append(k)
#         continue
#     if top is None:
#         top = model
#     if model is None:
#         model = top

#     traj_info[k] = {
#         "top": top,
#         "model": model,
#         "traj": sorted(trajs),
#     }
    
# print("systems:", len(traj_info))
# if missing:
#     print("missing top/model for:", len(missing), "keys (showing 10):", missing[:10])# === build traj_info dictionary (top, model, traj) ===

# # get all files from GCS directory
# traj_files  = [p for p in gs_ls(TRAJS_DIR)  if p.endswith((".xtc", ".trr", ".dcd"))]
# top_files   = [p for p in gs_ls(TOPOS_DIR)  if p.endswith((".psf", ".top", ".pdb"))]
# model_files = [p for p in gs_ls(MODELS_DIR) if p.endswith(".pdb")]

# # prepare containers for grouping files by system
# trajs = defaultdict(list)
# tops  = {}
# models  = {}

# for p in traj_files:
#     k = base3(p)
#     if k: trajs[k].append(p)

# def topo_rank(p):
#     if p.endswith(".psf"): return 0
#     if p.endswith(".top"): return 1
#     if p.endswith(".pdb"): return 2
#     return 99

# for p in top_files:
#     k = base3(p)
#     if not k:
#         continue
#     if (k not in tops) or (topo_rank(p) < topo_rank(tops[k])):
#         tops[k] = p

# for p in model_files:
#     k = base3(p)
#     if k and k not in models: models[k] = p

# traj_info = {}
# missing = []
# for k, trajs in trajs.items():
#     top = tops.get(k)
#     model = models.get(k)

#     # fallback: use whichever exists (model or top)
#     if top is None and model is None:
#         missing.append(k)
#         continue
#     if top is None:
#         top = model
#     if model is None:
#         model = top

#     traj_info[k] = {
#         "top": top,
#         "model": model,
#         "traj": sorted(trajs),
#     }
    
# print("systems:", len(traj_info))
# if missing:
#     print("missing top/model for:", len(missing), "keys (showing 10):", missing[:10])

In [4]:
# get all files from GCS directory
traj_files  = [p for p in gs_ls(TRAJS_DIR)  if p.endswith((".xtc", ".trr", ".dcd"))]
top_files   = [p for p in gs_ls(TOPOS_DIR)  if p.endswith((".psf", ".top", ".pdb"))]
model_files = [p for p in gs_ls(MODELS_DIR) if p.endswith(".pdb")]

# define ranking for topology preference based on whats available
def topo_rank(p):
    if p.endswith(".psf"): return 0
    if p.endswith(".top"): return 1
    if p.endswith(".pdb"): return 2
    return 99

# group by (protein~pdbid_chain, simID)
trajs = defaultdict(list)     # (base, simID) -> list of traj paths
tops  = {}                    # (base, simID) -> best topo path
models = {}                   # (base, simID) -> model pdb (optional)
traj_names = defaultdict(list)  # (base, simID) -> list of traj filenames (for printing)

for p in traj_files:
    b = base2(p); sid = sim_id(p)
    if b and sid:
        key = (b, sid)
        trajs[key].append(p)
        traj_names[key].append(Path(p).name)

for p in top_files:
    b = base2(p); sid = sim_id(p)
    if not (b and sid):
        continue
    key = (b, sid)
    if (key not in tops) or (topo_rank(p) < topo_rank(tops[key])):
        tops[key] = p

for p in model_files:
    b = base2(p); sid = sim_id(p)
    if b and sid and (b, sid) not in models:
        models[(b, sid)] = p

# build final traj_info
traj_info = {}
missing = []
for key, tlist in trajs.items():
    top = tops.get(key)
    model = models.get(key)

    if top is None and model is None:
        missing.append(key)
        continue
    if top is None:
        top = model
    if model is None:
        model = top

    traj_info[key] = {"top": top, "model": model, "traj": sorted(tlist)}

print("paired sims:", len(traj_info))
print("missing top/model for:", len(missing))
for (base, sid) in sorted(missing):
    names = traj_names.get((base, sid), [])
    if len(names) <= 3:
        traj_str = ", ".join(names)
    else:
        traj_str = ", ".join(names[:3]) + f", ... (+{len(names)-3} more)"
    print(f"{base} | simID={sid} | traj={traj_str}")

paired sims: 363
missing top/model for: 11
D(2)_dopamine_receptor~6CM4 | simID=1239 | traj=D(2)_dopamine_receptor~6CM4~unknown~19132_trj_1239.xtc
D(2)_dopamine_receptor~6CM4 | simID=1240 | traj=D(2)_dopamine_receptor~6CM4~unknown~19141_trj_1240.xtc
D(2)_dopamine_receptor~6CM4 | simID=1243 | traj=D(2)_dopamine_receptor~6CM4~1243~19164_trj_1243.xtc, D(2)_dopamine_receptor~6CM4~unknown~19164_trj_1243.xtc
D2~6CM4 | simID=1239 | traj=D2~6CM4~inactive~19129_trj_1239.xtc, D2~6CM4~inactive~19130_trj_1239.xtc, D2~6CM4~inactive~19131_trj_1239.xtc, ... (+1 more)
DRD2~6CM4 | simID=1239 | traj=DRD2~6CM4~inactive~19129_trj_1239.xtc
G(T)_subunit_beta-1~6VMS | simID=1241 | traj=G(T)_subunit_beta-1~6VMS~unknown~19148_trj_1241.xtc
G(T)_subunit_beta-1~6VMS | simID=1242 | traj=G(T)_subunit_beta-1~6VMS~1242~19157_trj_1242.xtc, G(T)_subunit_beta-1~6VMS~unknown~19157_trj_1242.xtc
Olfactory_receptor_51E2~8F76 | simID=1245 | traj=Olfactory_receptor_51E2~8F76~1245~19290_trj_1245.trr
Olfactory_receptor_51E2~8F76

## build protein systems

In [5]:
# === define universe for each system ===

warnings.filterwarnings("ignore", message="DCDReader currently makes independent timesteps", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="Reload offsets from trajectory")

MAX_SIMS = 50          # process 50 keys at a time
START_IDX = 337         # change this to resume (e.g., 50, 100, ...)

# clear cache before loading next batch
if CACHE_DIR.exists():
    shutil.rmtree(CACHE_DIR)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print("CACHE CLEARED")

universes = {}
skipped = []

items = list(traj_info.items())
end_idx = min(START_IDX + MAX_SIMS, len(items))

print(f"processing simulations {START_IDX} to {end_idx} out of {len(items)} total simulations")

for idx in tqdm(range(START_IDX, end_idx), desc="loading sims", unit="sim"):
    rec, info = items[idx]
    base, sid = rec
    tag = f"{base} | simID={sid}"
    
    try:
        top_local = gs_fetch(info["top"])
        top = str(top_local)

        for i, traj_gcs in enumerate(info["traj"], start=1):
            traj_local = gs_fetch(traj_gcs)
            traj_path = str(traj_local)

            print(f"processing {rec} rep {i}: {Path(traj_path).name}")
            u = mda.Universe(top, traj_path, in_memory=False)

            # store per-rep universe (key includes rep)
            universes[(rec, i)] = u

            prot = u.select_atoms("protein")
            ca   = u.select_atoms("protein and name CA")

            print(
                f"✅ {tag} rep {i:2d} | "
                f"frames: {len(u.trajectory):6d} | "
                f"total atoms: {u.atoms.n_atoms:6d} | "
                f"protein atoms: {prot.n_atoms:5d} | "
                f"CA atoms: {ca.n_atoms:4d} | "
                f"residues: {len(prot.residues):4d} | "
                f"top: {top_local.name}"
            )

    except Exception as e:
        skipped.append((idx, tag, str(e).splitlines()[0]))
        print(f"❌ {tag} | SKIPPED | {str(e).splitlines()[0]}")
        continue

CACHE CLEARED
processing simulations 337 to 363 out of 363 total simulations


loading sims:   0%|          | 0/26 [00:00<?, ?sim/s]

processing ('unknown~8T3S', '2283') rep 1: unknown~8T3S~unknown~25206_trj_2283.dcd
❌ unknown~8T3S | simID=2283 | SKIPPED | The topology and DCD trajectory files don't have the same number of atoms!
processing ('unknown~8UXY', '2057') rep 1: unknown~8UXY~unknown~24065_trj_2057.trr
❌ unknown~8UXY | simID=2057 | SKIPPED | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UXY~unknown~24064_dyn_2057.top with parser <class 'MDAnalysis.topology.TOPParser.TOPParser'>.
processing ('unknown~8UXY', '2069') rep 1: unknown~8UXY~unknown~24090_trj_2069.trr
❌ unknown~8UXY | simID=2069 | SKIPPED | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UXY~2069~24089_dyn_2069.top with parser <class 'MDAnalysis.topology.TOPParser.TOPParser'>.
processing ('unknown~8UYQ', '2061') rep 1: unknown~8UYQ~unknown~24081_trj_2061.trr
❌ unknown~8UYQ | simID=2061 | SKIPPED | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UYQ~unknown~24080_dyn_2061.top w

In [10]:
print("\n=== BATCH SUMMARY ===")
print(f"processed sim indices: [{START_IDX}:{end_idx})")
print(f"next START_IDX to resume: {end_idx}")
print(f"universes loaded this batch: {len(universes)}")
print(f"skipped sims this batch: {len(skipped)}")
if skipped:
    print("skipped (index | key | error):")
    for idx, tag, err in skipped:
        print(f"{idx:4d} | {tag} | {err}")


=== BATCH SUMMARY ===
processed sim indices: [337:363)
next START_IDX to resume: 363
universes loaded this batch: 0
skipped sims this batch: 26
skipped (index | key | error):
 337 | unknown~8T3S | simID=2283 | The topology and DCD trajectory files don't have the same number of atoms!
 338 | unknown~8UXY | simID=2057 | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UXY~unknown~24064_dyn_2057.top with parser <class 'MDAnalysis.topology.TOPParser.TOPParser'>.
 339 | unknown~8UXY | simID=2069 | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UXY~2069~24089_dyn_2069.top with parser <class 'MDAnalysis.topology.TOPParser.TOPParser'>.
 340 | unknown~8UYQ | simID=2061 | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~8UYQ~unknown~24080_dyn_2061.top with parser <class 'MDAnalysis.topology.TOPParser.TOPParser'>.
 341 | unknown~9CLW | simID=2297 | Failed to construct topology from file /home/jupyter/gcs_cache/unknown~9CLW~229

## extract early-segment features 

In [7]:
# === define analysis parameters ===

EARLY_FRAC = 0.50                    # early percentage of frames considered "early"
N_WINDOWS = None                     # number of windows to split the early segment into. if None, take all frames
ATOM_SEL   = "protein and name CA"   # for selecting CA atoms, works across all pdbs
SKIP_EQ_FRAMES = 100                 # discard first 100 frames as equilibration

In [8]:
# === helper function to compute early features ===

def rmsd_naive(A, B):
    diff = A - B
    return np.sqrt((diff * diff).sum(axis=1).mean())

# kabsch algorithm aligns two sets of points -> 
# in our case it removes overall translation and rotation so that the 
# metrics we compare are due to internal changes in the protein
def kabsch_align_centered(mobile, reference):
    # center the coordinates (remove translation)
    mobile_centered = mobile - mobile.mean(axis=0)
    reference_centered    = reference - reference.mean(axis=0)

    covariance = mobile_centered.T @ reference_centered  # covariance matrix
    V, _, Wt = np.linalg.svd(covariance) # optimal rotation using SVD

    # ensure proper rotation (right-handed coordinate system)
    if np.linalg.det(V @ Wt) < 0:
        V[:, -1] *= -1

    rotation = V @ Wt # optimal rotation matrix
    mobile_aligned_centered = mobile_centered @ rotation # rotate mobile coordinates
    return mobile_aligned_centered, reference_centered # translate back to reference center


def compute_early_features(
    u, receptor, rep_id,
    early_frac=0.20,
    n_windows=None,
    ca_sel="protein and name CA",
    tm3_sel="protein and name CA and resid 105:135",
    tm6_sel="protein and name CA and resid 250:280",
):
    # build selections
    ca = u.select_atoms(ca_sel)
    if ca.n_atoms == 0:
        raise ValueError(f"{receptor} rep {rep_id}: selection '{ca_sel}' returned 0 atoms")

    tm3 = u.select_atoms(tm3_sel)
    tm6 = u.select_atoms(tm6_sel)

    # define post-equilibration window (simulations sometimes have a really noise equilibriation period that skews results, so we want to skip it)
    n_frames = len(u.trajectory)
    start = min(SKIP_EQ_FRAMES, n_frames - 1)  # safety

    # define "early" frames
    n_early_frames = max(1, int(np.floor((n_frames - start) * early_frac)))

    # choose indicies of frames to calculate features on
    all_idxs = list(range(start, start + n_early_frames))
    if n_windows is None:  # use all early frames
        windows = [all_idxs]
    else:                  # split early segment into fixed windows
        windows = np.array_split(all_idxs, n_windows)

    # reference coordinates = first post-equilibration frame
    u.trajectory[start]
    ref = ca.positions.astype(np.float64).copy()

    # global per-frame values across the whole early segment (for mean/std/max)
    rmsd_all, rg_all, tm_all = [], [], []

    # per-frame time series (only used if n_windows is None)
    rmsd_ts, rg_ts, tm_ts = [], [], []
    frame_ts = []

    # windowed summaries
    rmsd_w, rg_w, tm_w = [], [], []
    rmsf_mean_w, rmsf_max_w = [], []

    for window in windows:
        rmsd_list, rg_list, tm_list = [], [], []
        aligned_coordinates = [] # needed for rmsf in this window

        for frame_idx in window:
            u.trajectory[frame_idx]
            coordinates = ca.positions.astype(np.float64)
            coordinates_aligned_centered, ref_centered = kabsch_align_centered(coordinates, ref)

            rmsd_val = rmsd_naive(coordinates_aligned_centered, ref_centered)
            rg_val = ca.radius_of_gyration()

            if tm3.n_atoms > 0 and tm6.n_atoms > 0:
                tm_val = np.linalg.norm(tm3.center_of_mass() - tm6.center_of_mass())
            else:
                tm_val = np.nan

            # collect full per-frame time series when n_windows is None
            if n_windows is None:
                rmsd_ts.append(float(rmsd_val))
                rg_ts.append(float(rg_val))
                tm_ts.append(float(tm_val))
                frame_ts.append(int(frame_idx))

            # collect window values
            rmsd_list.append(rmsd_val)
            rg_list.append(rg_val)
            tm_list.append(tm_val)

            # collect global values too (entire early segment)
            rmsd_all.append(rmsd_val)
            rg_all.append(rg_val)
            tm_all.append(tm_val)

            aligned_coordinates.append(coordinates_aligned_centered)

        # window means (time-aware coarse summaries)
        rmsd_w.append(np.nanmean(rmsd_list))
        rg_w.append(np.nanmean(rg_list))
        tm_w.append(np.nanmean(tm_list))

        # windowed rmsf only when n_windows is not None
        if n_windows is not None:
            Xw = np.stack(aligned_coordinates, axis=0)          # (n_win_frames, n_atoms, 3)
            Xw_mean = Xw.mean(axis=0)                           # (n_atoms, 3)
            rmsf_per_atom_w = np.sqrt(((Xw - Xw_mean) ** 2).sum(axis=2).mean(axis=0))  # (n_atoms,)

            rmsf_mean_w.append(float(np.nanmean(rmsf_per_atom_w)))
            rmsf_max_w.append(float(np.nanmax(rmsf_per_atom_w)))

    # global summaries over all early frames
    rmsd_arr = np.array(rmsd_all, dtype=float)
    rg_arr   = np.array(rg_all, dtype=float)
    tm_arr    = np.array(tm_all, dtype=float)

    out = {
        "receptor": receptor,
        "rep": rep_id,
        "n_frames_total": n_frames,
        "n_frames_used": len(all_idxs),
        "early_frac": early_frac,
        "n_windows": len(windows),

        "rmsd_mean": float(np.nanmean(rmsd_arr)),
        "rmsd_std":  float(np.nanstd(rmsd_arr)),
        "rmsd_max":  float(np.nanmax(rmsd_arr)),

        "rg_mean": float(np.nanmean(rg_arr)),
        "rg_std":  float(np.nanstd(rg_arr)),

        "tm3_tm6_mean": float(np.nanmean(tm_arr)),
        "tm3_tm6_std":  float(np.nanstd(tm_arr)),
    }

    # add windowed means
    if n_windows is not None:
        out.update({f"rmsd_w{i}": v for i, v in enumerate(rmsd_w)})
        out.update({f"rg_w{i}": v for i, v in enumerate(rg_w)})
        out.update({f"tm3_tm6_w{i}": v for i, v in enumerate(tm_w)})

    # add windowed rmsf summaries only if n_windows is not None
    if n_windows is not None:
        out.update({f"rmsf_mean_w{i}": v for i, v in enumerate(rmsf_mean_w)})
        out.update({f"rmsf_max_w{i}": v for i, v in enumerate(rmsf_max_w)})

    if n_windows is None:
        out["_timeseries"] = {
            "frame_idx": np.array(frame_ts, dtype=int),
            "rmsd": np.array(rmsd_ts, dtype=float),
            "rg": np.array(rg_ts, dtype=float),
            "tm3_tm6": np.array(tm_ts, dtype=float),
        }


    return out

In [9]:
# === build input features treating reps as independent rows ===

features = []

for (rec, i), u in tqdm(universes.items(), desc="computing features", unit="rep"):
    base, sid = rec
    print(f"processing {rec} rep {i}")

    feats = compute_early_features(
        u=u,
        receptor=base,
        rep_id=i,
        early_frac=EARLY_FRAC,
        n_windows=N_WINDOWS,
    )

    feats["simID"] = sid
    feats["rep"] = i

    # SAVE PER-FRAME METRICS IF PRESENT
    ts = feats.pop("_timeseries", None)
    if ts is not None:
        X = np.stack([ts["rmsd"], ts["rg"], ts["tm3_tm6"]], axis=1)
        safe_base = base.replace("~", "_")
        np.save(OUT_DIR / f"early_ts_{safe_base}_rep{i}_{sid}.npy", X)
        np.save(OUT_DIR / f"early_frames_{safe_base}_rep{i}_{sid}.npy", ts["frame_idx"])

    # metadata: now from the universe files, not traj_info in case we skipped any systems
    feats["traj_file"] = Path(u.trajectory.filename).name
    feats["top_file"]  = Path(u.filename).name  # topology filename

    features.append(feats)

features_df = pd.DataFrame(features).sort_values(["receptor", "rep"]).reset_index(drop=True)
features_df# === build input features treating reps as independent rows ===

features = []

for (rec, i), u in tqdm(universes.items(), desc="computing features", unit="rep"):
    base, sid = rec
    print(f"processing {rec} rep {i}")

    feats = compute_early_features(
        u=u,
        receptor=base,
        rep_id=i,
        early_frac=EARLY_FRAC,
        n_windows=N_WINDOWS,
    )

    feats["simID"] = sid
    feats["rep"] = i

    # SAVE PER-FRAME METRICS IF PRESENT
    ts = feats.pop("_timeseries", None)
    if ts is not None:
        X = np.stack([ts["rmsd"], ts["rg"], ts["tm3_tm6"]], axis=1)
        safe_base = base.replace("~", "_")
        np.save(OUT_DIR / f"early_ts_{safe_base}_rep{i}_{sid}.npy", X)
        np.save(OUT_DIR / f"early_frames_{safe_base}_rep{i}_{sid}.npy", ts["frame_idx"])

    # metadata: now from the universe files, not traj_info in case we skipped any systems
    feats["traj_file"] = Path(u.trajectory.filename).name
    feats["top_file"]  = Path(u.filename).name  # topology filename

    features.append(feats)

features_df = pd.DataFrame(features).sort_values(["receptor", "rep"]).reset_index(drop=True)
features_df

computing features: 0rep [00:00, ?rep/s]

KeyError: 'receptor'

computing features: 0rep [00:00, ?rep/s]

KeyError: 'receptor'

In [ ]:
# === sanity check we're saving the right number of frames ===

for (rec, i), u in universes.items():
    n_frames = len(u.trajectory)
    start = min(SKIP_EQ_FRAMES, n_frames - 1)
    n_early = int(np.floor((n_frames - start) * EARLY_FRAC))

    base, sid = rec
    safe_base = base.replace("~", "_")
    X = np.load(OUT_DIR / f"early_ts_{safe_base}_rep{i}_{sid}.npy")

    print(
        f"{rec} rep{i}: "
        f"expected={n_early}, "
        f"got={X.shape[0]}, "
        f"OK={X.shape == (n_early, 3)}"
    )# === sanity check we're saving the right number of frames ===

for (rec, i), u in universes.items():
    n_frames = len(u.trajectory)
    start = min(SKIP_EQ_FRAMES, n_frames - 1)
    n_early = int(np.floor((n_frames - start) * EARLY_FRAC))

    base, sid = rec
    safe_base = base.replace("~", "_")
    X = np.load(OUT_DIR / f"early_ts_{safe_base}_rep{i}_{sid}.npy")

    print(
        f"{rec} rep{i}: "
        f"expected={n_early}, "
        f"got={X.shape[0]}, "
        f"OK={X.shape == (n_early, 3)}"
    )

## label from full trajectories

In [ ]:
# === get the full trajectory ===

# takes ca atoms for each frame (after equilibration)
# aligns them to the first post-equilibration frame using kabsch algorithm
# returns array of shape (T, 3N) where T = number of frames, N = number of CA atoms
def extract_ca_coords_aligned(u, ca_sel="protein and name CA", skip_frames=0, dtype=np.float64):
    ca = u.select_atoms(ca_sel)
    if ca.n_atoms == 0:
        raise ValueError(f"CA selection '{ca_sel}' returned 0 atoms")

    n_frames = len(u.trajectory)
    start = min(skip_frames, n_frames - 1)

    # reference = ca coordinates at first post-equilibration frame
    u.trajectory[start]
    ref = ca.positions.astype(dtype).copy()

    coordinates = []

    # align using kabsch to remove translation and rotation wrt to the reference frame
    for _ in u.trajectory[start::1]:
        mobile = ca.positions.astype(dtype, copy=False)
        mobile_aligned_centered, _ = kabsch_align_centered(mobile, ref)  # returns centered+rotated
        coordinates.append(mobile_aligned_centered)

    X = np.asarray(coordinates, dtype=dtype)      # (T, N, 3)
    X = X.reshape(X.shape[0], -1)                 # (T, 3N)
    return X


In [ ]:
# === label full trajectory using TICA + k-means clustering ===

# for a given trajectory, convert into aligned coordinates,
# project into low dimensional TICA space to get the slow modes,
# cluster the frames in TICA space with k-means,
# computes (1) how dominant the biggest cluster is and (2) how well-separated the clusters are ("silhouette score"),
# determine multi-state vs single-state based on these metrics
def label_trajectory_tica(
    u,
    ca_sel="protein and name CA",
    lag=10,
    dim=2,
    n_clusters=2,
    dominant_thresh=0.85,
    silhouette_thresh=0.15,
    random_state=0,
):
    aligned_coordinates = extract_ca_coords_aligned(u, ca_sel=ca_sel, skip_frames=SKIP_EQ_FRAMES) # (T, 3N)
    T = aligned_coordinates.shape[0]

    # reduce dimensionality with tica to get the slow modes
    if T <= lag + 1: # tica needs at least lag+1 frames
        raise ValueError(f"not enough frames for lag={lag}!")
    tica = coor.tica(aligned_coordinates, lag=lag, dim=dim)
    traj_tica = tica.get_output()[0]  # (T, dim) trajectory projected into TICA space

    # cluster points in tica space
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
    labels = kmeans.fit_predict(traj_tica)

    # count how many frames are in each cluster -> determine how dominant the biggest cluster is
    counts = np.bincount(labels, minlength=n_clusters)
    fractions = counts / counts.sum()
    dominant = float(np.max(fractions))

    # compute silhouette score to assess cluster separation quality)
    # silhouette_score requires: at least 2 clusters present and each cluster has >= 2 samples
    silhouette = np.nan
    unique, unique_counts = np.unique(labels, return_counts=True)
    if len(unique) >= 2 and np.all(unique_counts >= 2):
        silhouette = float(silhouette_score(traj_tica, labels))
    # else: keep silhouette as nan bc there are too few points in a cluster to define it)

    # final label
    if ( # there is no clear dominant cluster and clusters are well-separated
        dominant <= dominant_thresh
        and (not np.isnan(silhouette))
        and silhouette >= silhouette_thresh
    ):
        traj_label = "multi_state"
    else:
        traj_label = "single_state"

    return {
        "traj_label": traj_label,
        "dominant_cluster_frac": dominant,
        "cluster_fracs": fractions,
        "silhouette": silhouette,
        "T_frames_used": int(T),
    }
# === label full trajectory using TICA + k-means clustering ===

# for a given trajectory, convert into aligned coordinates,
# project into low dimensional TICA space to get the slow modes,
# cluster the frames in TICA space with k-means,
# computes (1) how dominant the biggest cluster is and (2) how well-separated the clusters are ("silhouette score"),
# determine multi-state vs single-state based on these metrics
def label_trajectory_tica(
    u,
    ca_sel="protein and name CA",
    lag=10,
    dim=2,
    n_clusters=2,
    dominant_thresh=0.85,
    silhouette_thresh=0.15,
    random_state=0,
):
    aligned_coordinates = extract_ca_coords_aligned(u, ca_sel=ca_sel, skip_frames=SKIP_EQ_FRAMES) # (T, 3N)
    T = aligned_coordinates.shape[0]

    # reduce dimensionality with tica to get the slow modes
    if T <= lag + 1: # tica needs at least lag+1 frames
        raise ValueError(f"not enough frames for lag={lag}!")
    tica = coor.tica(aligned_coordinates, lag=lag, dim=dim)
    traj_tica = tica.get_output()[0]  # (T, dim) trajectory projected into TICA space

    # cluster points in tica space
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init="auto")
    labels = kmeans.fit_predict(traj_tica)

    # count how many frames are in each cluster -> determine how dominant the biggest cluster is
    counts = np.bincount(labels, minlength=n_clusters)
    fractions = counts / counts.sum()
    dominant = float(np.max(fractions))

    # compute silhouette score to assess cluster separation quality)
    # silhouette_score requires: at least 2 clusters present and each cluster has >= 2 samples
    silhouette = np.nan
    unique, unique_counts = np.unique(labels, return_counts=True)
    if len(unique) >= 2 and np.all(unique_counts >= 2):
        silhouette = float(silhouette_score(traj_tica, labels))
    # else: keep silhouette as nan bc there are too few points in a cluster to define it)

    # final label
    if ( # there is no clear dominant cluster and clusters are well-separated
        dominant <= dominant_thresh
        and (not np.isnan(silhouette))
        and silhouette >= silhouette_thresh
    ):
        traj_label = "multi_state"
    else:
        traj_label = "single_state"

    return {
        "traj_label": traj_label,
        "dominant_cluster_frac": dominant,
        "cluster_fracs": fractions,
        "silhouette": silhouette,
        "T_frames_used": int(T),
    }


In [ ]:
# === run full-trajectory labeling for each rep ===

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="Call to deprecated function \"tica\""
)

np.bool = bool  # workaround for pyemma using np.bool (numpy 2.0)

label_rows = []
skipped_labels = []  # keep track of skipped reps

for (rec, rep_i), u_full in tqdm(universes.items(), desc="labeling reps", unit="rep"):
    base, sid = rec
    print(f"✅ labeling {base} simID={sid} rep {rep_i}: {Path(u_full.trajectory.filename).name}")

    try:
        out = label_trajectory_tica(
            u_full,
            ca_sel=ATOM_SEL,
            lag=10,
            dim=2,
            n_clusters=2,
            dominant_thresh=0.75,
            silhouette_thresh=0.5,
            random_state=0,
        )

    except (ValueError, OSError) as e:
        print(f"❌ SKIPPED {base} simID={sid} rep {rep_i}: {str(e).splitlines()[0]}")
        skipped_labels.append((base, sid, rep_i, str(e).splitlines()[0]))
        continue

    except Exception as e:
        print(f"❌ SKIPPED {base} simID={sid} rep {rep_i}: {type(e).__name__}: {str(e).splitlines()[0]}")
        skipped_labels.append((base, sid, rep_i, f"{type(e).__name__}: {str(e).splitlines()[0]}"))
        continue

    label_rows.append({
        "receptor": base,
        "simID": sid,
        "rep": rep_i,
        "label": out["traj_label"],
        "dominant_cluster_frac": out["dominant_cluster_frac"],
        "silhouette": out["silhouette"],
        "n_frames": out["T_frames_used"],
        "traj_file": Path(u_full.trajectory.filename).name,
        "top_file": Path(u_full.filename).name,
    })

print(f"\n SUMMARY: skipped {len(skipped_labels)} reps due to insufficient frames")
# === run full-trajectory labeling for each rep ===

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message="Call to deprecated function \"tica\""
)

np.bool = bool  # workaround for pyemma using np.bool (numpy 2.0)

label_rows = []
skipped_labels = []  # keep track of skipped reps

for (rec, rep_i), u_full in tqdm(universes.items(), desc="labeling reps", unit="rep"):
    base, sid = rec
    print(f"✅ labeling {base} simID={sid} rep {rep_i}: {Path(u_full.trajectory.filename).name}")

    try:
        out = label_trajectory_tica(
            u_full,
            ca_sel=ATOM_SEL,
            lag=10,
            dim=2,
            n_clusters=2,
            dominant_thresh=0.75,
            silhouette_thresh=0.5,
            random_state=0,
        )

    except (ValueError, OSError) as e:
        print(f"❌ SKIPPED {base} simID={sid} rep {rep_i}: {str(e).splitlines()[0]}")
        skipped_labels.append((base, sid, rep_i, str(e).splitlines()[0]))
        continue

    except Exception as e:
        print(f"❌ SKIPPED {base} simID={sid} rep {rep_i}: {type(e).__name__}: {str(e).splitlines()[0]}")
        skipped_labels.append((base, sid, rep_i, f"{type(e).__name__}: {str(e).splitlines()[0]}"))
        continue

    label_rows.append({
        "receptor": base,
        "simID": sid,
        "rep": rep_i,
        "label": out["traj_label"],
        "dominant_cluster_frac": out["dominant_cluster_frac"],
        "silhouette": out["silhouette"],
        "n_frames": out["T_frames_used"],
        "traj_file": Path(u_full.trajectory.filename).name,
        "top_file": Path(u_full.filename).name,
    })

print(f"\n SUMMARY: skipped {len(skipped_labels)} reps due to insufficient frames")


In [ ]:
# === build + save base dataset (one row per receptor/rep) ===

labels_df = pd.DataFrame(label_rows)

df = features_df.merge(
    labels_df[["receptor", "simID", "rep", "label", "dominant_cluster_frac", "silhouette"]],
    on=["receptor", "simID", "rep"],
    how="left",
)

# numeric target (0 or 1)
label_to_y = {"single_state": 0, "multi_state": 1}
df["y"] = df["label"].map(label_to_y)

# paths to saved early time series (include simID so filenames are unique)
df["early_ts_path"] = df.apply(
    lambda r: f"early_ts_{r.receptor.replace('~','_')}_rep{int(r.rep)}_{r.simID}.npy",
    axis=1,
)
df["early_frames_path"] = df.apply(
    lambda r: f"early_frames_{r.receptor.replace('~','_')}_rep{int(r.rep)}_{r.simID}.npy",
    axis=1,
)

# keep only labeled + files exist
df = df[df["label"].notna()].copy()
df = df[
    df["early_ts_path"].apply(lambda p: (OUT_DIR / p).exists())
    & df["early_frames_path"].apply(lambda p: (OUT_DIR / p).exists())
].copy()

n_traj = df[["receptor", "simID", "rep"]].drop_duplicates().shape[0]
n_pos = df["y"].sum()
n_neg = n_traj - n_pos

print(f"trajectories saved: {n_traj}")
print(f"multi_state: {n_pos} | single_state: {n_neg}")

base_csv = OUT_DIR / "base_dataset.csv"
if base_csv.exists():
    df.to_csv(base_csv, mode="a", header=False, index=False)
else:
    df.to_csv(base_csv, index=False)

print("saved:", base_csv)

df# === build + save base dataset (one row per receptor/rep) ===

labels_df = pd.DataFrame(label_rows)

df = features_df.merge(
    labels_df[["receptor", "simID", "rep", "label", "dominant_cluster_frac", "silhouette"]],
    on=["receptor", "simID", "rep"],
    how="left",
)

# numeric target (0 or 1)
label_to_y = {"single_state": 0, "multi_state": 1}
df["y"] = df["label"].map(label_to_y)

# paths to saved early time series (include simID so filenames are unique)
df["early_ts_path"] = df.apply(
    lambda r: f"early_ts_{r.receptor.replace('~','_')}_rep{int(r.rep)}_{r.simID}.npy",
    axis=1,
)
df["early_frames_path"] = df.apply(
    lambda r: f"early_frames_{r.receptor.replace('~','_')}_rep{int(r.rep)}_{r.simID}.npy",
    axis=1,
)

# keep only labeled + files exist
df = df[df["label"].notna()].copy()
df = df[
    df["early_ts_path"].apply(lambda p: (OUT_DIR / p).exists())
    & df["early_frames_path"].apply(lambda p: (OUT_DIR / p).exists())
].copy()

n_traj = df[["receptor", "simID", "rep"]].drop_duplicates().shape[0]
n_pos = df["y"].sum()
n_neg = n_traj - n_pos

print(f"trajectories saved: {n_traj}")
print(f"multi_state: {n_pos} | single_state: {n_neg}")

base_csv = OUT_DIR / "base_dataset.csv"
if base_csv.exists():
    df.to_csv(base_csv, mode="a", header=False, index=False)
else:
    df.to_csv(base_csv, index=False)

print("saved:", base_csv)

df

In [ ]:
print("df.shape:", df.shape)  # true row count
print("unique (receptor,rep):", df[["receptor","rep"]].drop_duplicates().shape[0])
print("duplicates:", df[["receptor","rep"]].duplicated().sum())

## Save Data to Google Cloud Storage

In [ ]:
destination_dir = f"{BUCKET_DIR}/data/processed/v3"
processed_data_local_path = OUT_DIR 

dry_run = True

if dry_run:
    # 1. Preview what will be uploaded (no hidden files)
    !gsutil -m rsync -r -n -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}
else:
    #When ready to actually upload removes the -n (no-op) flag
    !gsutil -m rsync -r -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}destination_dir = f"{BUCKET_DIR}/data/processed/v3"
processed_data_local_path = OUT_DIR 

dry_run = True

if dry_run:
    # 1. Preview what will be uploaded (no hidden files)
    !gsutil -m rsync -r -n -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}
else:
    #When ready to actually upload removes the -n (no-op) flag
    !gsutil -m rsync -r -x '(^|\/)\..*' {processed_data_local_path} {destination_dir}